In [ ]:
## Setup — packages & environment

import sys, subprocess
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

required = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'networkx', 'openpyxl', 'reportlab']
for pkg in required:
    try:
        __import__(pkg.split('-')[0])
    except:
        install(pkg)

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import networkx as nx
import os, datetime as dt

RSEED = 2023
np.random.seed(RSEED)
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

In [ ]:
try:
    import ipykernel
    print('ipykernel installed')
except:
    print('ipykernel not installed')

In [ ]:
## 1. Temporal Network Data

# Create synthetic temporal network data (week-by-week)
np.random.seed(RSEED)
n_students = 25
n_weeks = 8

temporal_edges = []
for week in range(1, n_weeks + 1):
    # Add edges with increasing participation
    n_edges = 10 + week * 3
    for _ in range(n_edges):
        u = np.random.randint(0, n_students)
        v = np.random.randint(0, n_students)
        if u != v:
            temporal_edges.append({
                'week': week,
                'source': f'student_{u}',
                'target': f'student_{v}',
                'weight': np.random.randint(1, 3)
            })

df = pd.DataFrame(temporal_edges)
print('Temporal Network Data:')
print(df.head())
print(f'\nEdges per week:')
print(df.groupby('week').size())

In [ ]:
## 2. Time-Sliced Networks and Dynamic Metrics

# Build networks for each week
networks = {}
degree_evolution = {}
density_evolution = []

for week in range(1, n_weeks + 1):
    week_edges = df[df['week'] <= week]
    G = nx.Graph()
    for _, row in week_edges.iterrows():
        G.add_edge(row['source'], row['target'], weight=row['weight'])
    
    networks[week] = G
    density_evolution.append(nx.density(G))
    
    if week == 1:
        degree_evolution = {node: [] for node in G.nodes()}
    
    for node in G.nodes():
        if node not in degree_evolution:
            degree_evolution[node] = [0] * (week - 1)
        degree_evolution[node].append(G.degree(node))
    
    # Pad missing nodes with 0
    for node in degree_evolution:
        if len(degree_evolution[node]) < week:
            degree_evolution[node].append(0)

print('Network Evolution Summary:')
for week, G in networks.items():
    print(f'Week {week}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, density={nx.density(G):.3f}')

In [ ]:
## 3. Visualizations

os.makedirs('figures', exist_ok=True)

# 1. Network density evolution
fig, ax = plt.subplots(figsize=(10, 6))
weeks = range(1, n_weeks + 1)
ax.plot(weeks, density_evolution, 'o-', linewidth=2.5, markersize=8, color='steelblue')
ax.fill_between(weeks, density_evolution, alpha=0.3, color='steelblue')
ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Network Density', fontsize=12)
ax.set_title('Network Density Evolution Over Time', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/01_density_evolution.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 01_density_evolution.png')

# 2. Node count evolution
fig, ax = plt.subplots(figsize=(10, 6))
node_counts = [networks[w].number_of_nodes() for w in weeks]
edge_counts = [networks[w].number_of_edges() for w in weeks]
ax.plot(weeks, node_counts, 'o-', linewidth=2.5, markersize=8, label='Nodes', color='green')
ax.plot(weeks, edge_counts, 's-', linewidth=2.5, markersize=8, label='Edges', color='orange')
ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Network Growth: Nodes and Edges', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/02_growth_evolution.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 02_growth_evolution.png')

# 3. Degree trajectories for top nodes
fig, ax = plt.subplots(figsize=(10, 6))
final_degrees = {node: degree_evolution[node][-1] for node in degree_evolution}
top_nodes = sorted(final_degrees.items(), key=lambda x: x[1], reverse=True)[:5]
colors = sns.color_palette('Set2', 5)
for (node, _), color in zip(top_nodes, colors):
    ax.plot(weeks, degree_evolution[node], 'o-', linewidth=2, markersize=6, label=node, color=color)
ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Degree', fontsize=12)
ax.set_title('Degree Trajectories: Top 5 Most Connected Students', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/03_degree_trajectories.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 03_degree_trajectories.png')

df.to_csv('temporal_network_edges.csv', index=False)
print('\nSaved temporal network data')

In [ ]:
## 4. PDF Handout Generation

from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak
from reportlab.lib.enums import TA_JUSTIFY

pdf_path = 'Ch17_TemporalNetworks_Handout.pdf'
doc = SimpleDocTemplate(pdf_path, pagesize=letter, rightMargin=0.75*inch, leftMargin=0.75*inch,
                        topMargin=0.75*inch, bottomMargin=0.75*inch)

styles = getSampleStyleSheet()
styleN = styles['Normal']
styleN.fontSize = 11
styleN.alignment = TA_JUSTIFY

story = []
story.append(Paragraph('<b>Chapter 17: Temporal Network Analysis</b>', styles['Heading1']))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>1. Introduction</b>', styles['Heading2']))
intro = (
    'Temporal networks track how relationships evolve over time. '
    'Week-by-week analysis reveals network dynamics, including growth phases and stabilization.'
)
story.append(Paragraph(intro, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>2. Methods</b>', styles['Heading2']))
methods = (
    f'Time-sliced networks computed for weeks 1-{n_weeks}. '
    'Network density, size, and node degree centrality tracked over time.'
)
story.append(Paragraph(methods, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>3. Results</b>', styles['Heading2']))
story.append(Spacer(1, 6))

try:
    if os.path.exists('figures/01_density_evolution.png'):
        story.append(Image('figures/01_density_evolution.png', width=480, height=300))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 1: Network density trajectory.', styleN))
        story.append(Spacer(1, 12))
except: pass

try:
    if os.path.exists('figures/02_growth_evolution.png'):
        story.append(Image('figures/02_growth_evolution.png', width=480, height=300))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 2: Network size growth.', styleN))
        story.append(Spacer(1, 12))
except: pass

story.append(PageBreak())
story.append(Paragraph('<b>4. Interpretation</b>', styles['Heading2']))
interp = (
    'Network density increase indicates growing interaction. '
    'Individual degree trajectories reveal participation patterns—accelerating or declining engagement.'
)
story.append(Paragraph(interp, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph(f'Generated on: {dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', styleN))

try:
    doc.build(story)
    print(f'Saved PDF: {pdf_path}')
except Exception as e:
    print(f'PDF generation failed: {e}')